In [1]:
SYSTEM_PROMPT = """
Bạn là một hệ thống trích xuất span từ bệnh án tiếng Việt.

## NHIỆM VỤ

Đọc toàn bộ văn bản.

Liệt kê tất cả các span liên quan đến y khoa xuất hiện trong văn bản.
Chỉ trích xuất một span biểu diễn một khái niệm y khoa. Span có thể là một từ hoặc một cụm từ liên tiếp.
Span là chuỗi ký tự liên tiếp xuất hiện nguyên văn trong văn bản.
Không được thay đổi bất kỳ ký tự nào của span.

Mục tiêu quan trọng nhất là **không bỏ sót**.

## SPAN LIÊN QUAN ĐẾN Y KHOA

Bao gồm nhưng không giới hạn:

* bệnh
* chẩn đoán
* triệu chứng
* dấu hiệu
* thuốc
* hoạt chất
* xét nghiệm
* kết quả xét nghiệm
* thủ thuật
* cơ quan giải phẫu
* vi sinh vật
* chỉ số sinh học
* thuật ngữ y khoa

## QUY TẮC

Chỉ sao chép nguyên văn từ văn bản.

Không được:

* diễn giải
* chuẩn hóa
* dịch
* viết lại
* thêm từ
* bớt từ

Không trích xuất:

* hành động
* diễn biến
* câu mô tả
* trạng thái chung
* tiêu đề

Không trích xuất các từ chỉ vai trò hoặc tiêu đề như:

* triệu chứng
* chẩn đoán
* đánh giá
* bệnh sử
* tiền sử
* kết quả
* lý do nhập viện

Trừ khi chính chúng là một phần của thuật ngữ y khoa.

Mỗi dòng chỉ chứa MỘT thực thể y khoa.

Luôn chọn span ngắn nhất vẫn giữ nguyên ý nghĩa y khoa.

Không bao gồm các từ xung quanh không thuộc thực thể.

Nếu trong một cụm có nhiều thực thể thì phải tách riêng.

Lặp lại nhiều lượt.

Ở mỗi lượt, hãy tiếp tục tìm các thực thể chưa được liệt kê.

Chỉ kết thúc khi không còn thực thể y khoa nào chưa được liệt kê.

Ví dụ

Sai

bệnh nhân sốt đến 38.8°C

Đúng

sốt

38.8°C

----------------

Sai

công thức máu (cbc) nâng cao lên 11.3

Đúng

công thức máu (cbc)

11.3

----------------

Sai

ho và mệt mỏi

Đúng

ho

mệt mỏi

----------------

Sai

chụp x-quang ngực không phát hiện viêm phổi hoặc phù phổi

Đúng

chụp x-quang ngực

viêm phổi

phù phổi

---

Không ghép hai span không liền nhau.

Không tạo span mới.

Nếu phân vân, hãy ưu tiên giữ lại.

## KHÔNG TRÍCH XUẤT

* số thứ tự
* tiêu đề
* tên mục
* dấu câu
* khoảng trắng
* thông tin hành chính
* tên bác sĩ
* tên bệnh viện
* địa chỉ
* thông tin không mang ý nghĩa y khoa

## ĐỊNH DẠNG ĐẦU RA

Chỉ trả về danh sách các span.

Mỗi dòng đúng một span.

Không đánh số.

Không thêm ký hiệu đầu dòng.

Không giải thích.

Không thêm bất kỳ văn bản nào ngoài các span.

"""

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-8B"
)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [3]:
from pathlib import Path

DATA_DIR = Path(
    "/kaggle/input/datasets/kangaonkaggle/viettel-ai-race-2026-data/Ontological Reasoning - 1st Round"
)

files = sorted(DATA_DIR.glob("*.txt"))

print(len(files))

100


In [4]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-8B",
    device_map="auto"
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [5]:
from pathlib import Path
import json
import re
from tqdm.auto import tqdm

OUT_DIR = Path("/kaggle/working/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def ask_qwen(text):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": text,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    print("=" * 80)
    print("PROMPT SENT TO MODEL")
    print("=" * 80)
    print(prompt)
    
    device = next(model.parameters()).device
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    print(f"Input characters : {len(text)}")
    print(f"Input tokens     : {inputs.input_ids.shape[1]}")

    outputs = model.generate(
        **inputs,
        max_new_tokens=4096,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    print("Generated tokens:", len(generated_ids))

    
    print("="*80)
    
    print(f"OUTPUTS SHAPE")
    print(outputs.shape)
    
    print(f"INPUTS SHAPE")
    print(inputs.input_ids.shape)
    print("=" * 80)
    
    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=False,
    )

    print("=" * 80)
    print("MODEL OUTPUT")
    print("=" * 80)
    print(response)


    print("="*80)
    print(f"LAST 300")
    print("="*80)
    print(repr(response[-300:]))
    
    return response

In [6]:
from pathlib import Path

file = Path(
    "/kaggle/input/datasets/kangaonkaggle/viettel-ai-race-2026-data/"
    "Ontological Reasoning - 1st Round/32.txt"
)

text = file.read_text(encoding="utf-8")

print("=" * 80)
print("INPUT")
print("=" * 80)
print(text)

print("\n" + "=" * 80)
print("QWEN OUTPUT")
print("=" * 80)

response = ask_qwen(text)

INPUT
1. Tiền sử bệnh nội
Các bệnh lý mạn tính
- ho Bệnh bạch cầu dòng tủy mãn tính
- Tăng huyết áp
- Đái tháo đường típ 2
- ho Rung nhĩ kèm đáp ứng thất nhanh
- bệnh thận mạn, không đặc hiệu
2. Bệnh sử hiện tại
Lý do vào viện: Khó thở tăng dần, diễn tiến nặng hơn từ sáng ngày nhập viện.
Khoảng 5 ngày trước nhập viện, bệnh nhân xuất hiện cảm giác nghẹt ngực kèm sốt, nhiệt độ cao nhất khoảng 38,3°C. 
Sau đó không còn sốt nhưng tình trạng nghẹt ngực vẫn kéo dài. 
Trong vài ngày gần đây, bệnh nhân xuất hiện khó thở tăng dần, chủ yếu khi gắng sức, kèm phù ngoại vi tăng dần trong vài tuần trở lại đây.
Bệnh nhân có tiền sử nhập viện gần đây vì sốt và đau vai. 
Trong đợt điều trị đó, bệnh nhân diễn tiến suy hô hấp kèm tăng huyết áp, không đáp ứng đáng kể với điều trị lợi tiểu và phải chuyển khoa Hồi sức tích cực để hỗ trợ thở áp lực dương không xâm nhập (BiPAP). 
Tình trạng hô hấp cải thiện sau hỗ trợ hô hấp và bệnh nhân ổn định trở lại. 
Trong thời gian nằm viện, bệnh nhân được chẩn đoán nhi

In [7]:
import shutil
from pathlib import Path

zip_path = "/kaggle/working/output.zip"

shutil.make_archive(
    "/kaggle/working/output",
    "zip",
    root_dir=OUT_DIR
)

print(zip_path)

/kaggle/working/output.zip
